<a href="https://colab.research.google.com/github/PDLNABIN/Chandra-GGUF/blob/master/Summerization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install beautifulsoup4 openai

In [25]:
import re
from bs4 import BeautifulSoup

In [26]:
INPUT_MD_FILE  = "complete_ocr.md"
OUTPUT_MD_FILE = "complete_ocr_kv.md"

SKIP_LABELS = {"क्र. सं.", "क्र.सं.", "क्र.स.", "क्र.स", "क्र. स.",
               "क्र. स", "क्र सं.", "क्र सं", "क्र.सं", "सिं नं.",
               "सिं.नं.", "सिं.नं", "सि.नं.", "s.n.", "sn", "no.", "क्र"}

In [27]:
with open(INPUT_MD_FILE, "r", encoding="utf-8") as f:
    md_content = f.read()

print(f"{len(md_content):,} characters loaded")

775,044 characters loaded


In [28]:
def extract_all_tables(text):
    pattern = re.compile(r'<table[\s\S]*?</table>', re.IGNORECASE)
    return [(m.start(), m.end(), m.group()) for m in pattern.finditer(text)]

all_tables = extract_all_tables(md_content)
print(f"Found {len(all_tables)} tables")

Found 454 tables


In [29]:
def parse_grid(html):
    soup = BeautifulSoup(html, "html.parser")
    table = soup.find("table")

    grid, is_header = {}, {}

    for ri, tr in enumerate(table.find_all("tr")):
        ci = 0
        for cell in tr.find_all(["th", "td"]):
            while (ri, ci) in grid:
                ci += 1

            text = cell.get_text(" ", strip=True)
            rs = int(cell.get("rowspan", 1))
            cs = int(cell.get("colspan", 1))
            hdr = (cell.name == "th")

            for r in range(rs):
                for c in range(cs):
                    grid[(ri + r, ci + c)] = text
                    is_header[(ri + r, ci + c)] = hdr

            ci += cs

    n_rows = max(r for r, c in grid) + 1
    n_cols = max(c for r, c in grid) + 1

    rows = [[grid.get((r, c), "") for c in range(n_cols)] for r in range(n_rows)]
    hdrs = [[is_header.get((r, c), False) for c in range(n_cols)] for r in range(n_rows)]

    return rows, hdrs, n_rows, n_cols

In [33]:
def table_to_kv_blocks(html):
    rows, hdrs, n_rows, n_cols = parse_grid(html)

    # Detect header rows
    header_end = 0
    for ri in range(n_rows):
        if all(hdrs[ri]):
            header_end = ri + 1
        else:
            break

    # ========================================================
    # CASE 1: NORMAL TABLE (HAS HEADERS)
    # ========================================================
    if header_end > 0:
        labels = []
        for ci in range(n_cols):
            parts = []
            for ri in range(header_end):
                val = rows[ri][ci]
                if val and val not in parts:
                    parts.append(val)

            label = " / ".join(parts).strip()

            if label.lower() in {s.lower() for s in SKIP_LABELS}:
                labels.append(None)
            else:
                labels.append(label)

        valid_cols = [i for i, l in enumerate(labels) if l]

        if not valid_cols:
            return "[तालिकामा डेटा उपलब्ध छैन।]"

        clean_labels = [labels[i] for i in valid_cols]

        blocks = []

        for ri in range(header_end, n_rows):
            pairs = []

            for ci, label in zip(valid_cols, clean_labels):
                value = rows[ri][ci].strip()
                if not value:
                    continue

                pairs.append(f"{label}: {value}")

            if pairs:
                block = "  \n".join(pairs)   # Markdown-safe line break
                blocks.append(block)

        return "\n\n".join(blocks)

    # ========================================================
    # CASE 2: NO HEADER TABLE (LIKE YOUR EXAMPLE)
    # ========================================================
    else:
        items = []

        for ri in range(n_rows):
            for ci in range(n_cols):
                val = rows[ri][ci].strip()

                if not val:
                    continue

                # remove numbering like "१.", "2.", etc.
                val = re.sub(r'^\s*[\d०-९]+[.)]\s*', '', val)

                items.append(val)

        if not items:
            return "[तालिकामा डेटा उपलब्ध छैन।]"

        # Each item on its own line
        return "  \n".join(items)

In [34]:
tables_reversed = sorted(all_tables, key=lambda x: x[0], reverse=True)

modified_md = md_content
converted   = 0
errors      = 0

for idx, (start, end, html) in enumerate(tables_reversed):
    table_num = len(tables_reversed) - idx

    print(f"Table {table_num:>4}/{len(all_tables)} — Converting...",
          end=" ", flush=True)

    try:
        kv_text = table_to_kv_blocks(html)

        # spacing for readability
        kv_text = "\n\n" + kv_text + "\n\n"

        modified_md = modified_md[:start] + kv_text + modified_md[end:]
        converted += 1

        print("Done.")

    except Exception as e:
        errors += 1
        print(f"ERROR: {e}")

Table  454/454 — Converting... Done.
Table  453/454 — Converting... Done.
Table  452/454 — Converting... Done.
Table  451/454 — Converting... Done.
Table  450/454 — Converting... Done.
Table  449/454 — Converting... Done.
Table  448/454 — Converting... Done.
Table  447/454 — Converting... Done.
Table  446/454 — Converting... Done.
Table  445/454 — Converting... Done.
Table  444/454 — Converting... Done.
Table  443/454 — Converting... Done.
Table  442/454 — Converting... Done.
Table  441/454 — Converting... Done.
Table  440/454 — Converting... Done.
Table  439/454 — Converting... Done.
Table  438/454 — Converting... Done.
Table  437/454 — Converting... Done.
Table  436/454 — Converting... Done.
Table  435/454 — Converting... Done.
Table  434/454 — Converting... Done.
Table  433/454 — Converting... Done.
Table  432/454 — Converting... Done.
Table  431/454 — Converting... Done.
Table  430/454 — Converting... Done.
Table  429/454 — Converting... Done.
Table  428/454 — Converting... Done.
T

In [35]:
with open(OUTPUT_MD_FILE, "w", encoding="utf-8") as f:
    f.write(modified_md)

